In [ ]:
# Dataset > Feature Extraction > ML Model > Predictions (Driven by the LLM)

In [ ]:
!pip install pandas numpy scikit-learn xgboost langchain langchain-community ollama


#NOTE: idk if it will export the outputs too. if it doesn't or if you'd just like to run it yourself, this cell should download all required packages
#BUT you will need to download ollama 
#to do so run the following command in Powershell: irm https://ollama.com/install.ps1 | iex
#it'll take some time to download but then run the following commands:
            #ollama pull llama3
            #ollama run llama3

#if you do not have a windows device just look up ollama download 


In [6]:
import os
import pickle

DATA_PATH = "WESAD"

#My .pkl files were in WESAD/S#.pkl from the code > may need to be changed 
subjects = [f for f in os.listdir(DATA_PATH) if f.endswith(".pkl")]

all_data = []

for file_name in subjects:
    file_path = os.path.join(DATA_PATH, file_name)
    
    with open(file_path, "rb") as f:
        data = pickle.load(f, encoding="latin1")
    
    wrist = data["signal"]["wrist"]
    
    acc = wrist["ACC"]
    bvp = wrist["BVP"]
    eda = wrist["EDA"]
    temp = wrist["TEMP"]
    
    labels = data["label"]
    
    all_data.append({
        "subject": file_name.replace(".pkl", ""),
        "acc": acc,
        "bvp": bvp,
        "eda": eda,
        "temp": temp,
        "labels": labels
    })

print(f"Loaded {len(all_data)} subjects")

Loaded 15 subjects


In [17]:
def extract_features(signal, labels, window_size=40, step=40):
    rows = []
    
    # Had to look up for help here, but needed to scale to match the data
    factor = int(len(labels) / len(signal["eda"]))
    labels_ds = labels[::factor]
    
    min_len = min(len(labels_ds), len(signal["eda"]))
    
    for i in range(0, min_len - window_size, step):
        
        label_window = labels_ds[i:i+window_size]
        label = np.bincount(label_window.astype(int)).argmax()
        
        # Keep only emotion labels
        if label not in [1,2,3,4]:
            continue
        
        eda = signal["eda"][i:i+window_size]
        temp = signal["temp"][i:i+window_size]
        
        # Had to look up for help here as well, but needed to scale to match the data
        acc = signal["acc"][i*8:(i+window_size)*8]  
        bvp = signal["bvp"][i*16:(i+window_size)*16]
        
        # Safety check
        if len(eda) == 0 or len(temp) == 0 or len(acc) == 0 or len(bvp) == 0:
            continue
        
        row = {
            "eda_mean": np.mean(eda),
            "eda_std": np.std(eda),
            "temp_mean": np.mean(temp),
            "acc_mean": np.mean(acc),
            "acc_std": np.std(acc),
            "bvp_mean": np.mean(bvp),
            "bvp_std": np.std(bvp),
            "label": label
        }
        
        if any(np.isnan(v) for v in row.values()):
            continue
        
        rows.append(row)
    
    return rows

In [27]:
dataset = []

#sets data to the variabl
for subj in all_data:
    signal_dict = {
        "eda": subj["eda"],
        "acc": subj["acc"],
        "temp": subj["temp"],
        "bvp": subj["bvp"]
    }
    
    rows = extract_features(signal_dict, subj["labels"])
    dataset.extend(rows)

#df = df.groupby("label").sample(n=5000, random_state=42)
df = pd.DataFrame(dataset)

print(df.head())
print("Total samples:", len(df))

   eda_mean   eda_std  temp_mean   acc_mean    acc_std  bvp_mean    bvp_std  \
0  0.403612  0.061010    33.2535  27.027083  24.489226 -0.200656  77.102798   
1  0.376461  0.006709    33.2835  29.389583  21.479212  0.720672  81.039538   
2  0.452380  0.056718    33.3120  29.655208  20.875596 -0.147172  36.581211   
3  0.408344  0.017809    33.3240  28.875000  22.212773  1.528188  79.957260   
4  0.378732  0.004508    33.3220  27.911458  23.402846 -1.729531  84.188990   

   label  
0      1  
1      1  
2      1  
3      1  
4      1  
Total samples: 4496


In [28]:
#links label to emotion
label_map = {
    1: "baseline",
    2: "stress",
    3: "amusement",
    4: "meditation"
}

df["emotion"] = df["label"].map(label_map)

In [29]:
from sklearn.model_selection import train_test_split

#X contains all the data except for the labels, y contains the labels
X = df.drop(columns=["label", "emotion"])
y = df["emotion"]

#train/test split of the resulting sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [30]:
#I used XGBClassifier because I saw it recommended for this project, but if you want to explore with other models you just need to change this block
#I will try to make a notebook that compares different model accuracy later if I can and we're still looking for a model/algorithm

from xgboost import XGBClassifier
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
y_train_enc = le.fit_transform(y_train)
y_test_enc = le.transform(y_test)

model = XGBClassifier(
    n_estimators=100,
    max_depth=5,
    learning_rate=0.1,
    objective="multi:softprob",
    eval_metric="mlogloss"
)

model.fit(X_train, y_train_enc)

,objective,'multi:softprob'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,None
,device,None
,early_stopping_rounds,None
,enable_categorical,False
,eval_metric,'mlogloss'


In [31]:
import numpy as np
#this code gets the predicted labels and how accurate they are


probs = model.predict_proba(X_test)
y_pred_ml = np.argmax(probs, axis=1)

y_pred_labels = le.inverse_transform(y_pred_ml)

In [32]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

print("ML Accuracy:", accuracy_score(y_test, y_pred_labels))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_labels))

print("\nClassification Report:")
print(classification_report(y_test, y_pred_labels))

#currently the model is 97.77% accurate but this does not involve LLMs to this point so idk if this is what we're looking for???

ML Accuracy: 0.9777777777777777

Confusion Matrix:
[[109   2   1   0]
 [  1 335   1   6]
 [  1   2 255   2]
 [  0   3   1 181]]

Classification Report:
              precision    recall  f1-score   support

   amusement       0.98      0.97      0.98       112
    baseline       0.98      0.98      0.98       343
  meditation       0.99      0.98      0.98       260
      stress       0.96      0.98      0.97       185

    accuracy                           0.98       900
   macro avg       0.98      0.98      0.98       900
weighted avg       0.98      0.98      0.98       900



In [34]:
#langchain sets the LLM to Ollama - I used this because it was free, open source, and easy to set up. I can't tell how great it is as an LLM
#We may consider bringing up changing the LLM we use if that's part of Majumder's expectations as well
from langchain_community.llms import Ollama

llm = Ollama(model="llama3")

In [39]:
#I actually don't use this method ever. I had it combined with another langchain system that would allow me to ask several models at once
#but it looks like it was removed or changed in this version of langchain, I am still trying to find a way to make it work

from langchain_core.prompts import PromptTemplate

template = """
You are an expert in emotion recognition using physiological signals.

Features:
EDA mean: {eda_mean}
EDA std: {eda_std}
BVP mean: {bvp_mean}
BVP std: {bvp_std}
Temp mean: {temp_mean}
ACC mean: {acc_mean}
ACC std: {acc_std}

ML model predicted: {ml_prediction}

Based on physiological patterns:
- Stress → high HR, high EDA
- Amusement → moderate HR, variable EDA
- Meditation → low HR, stable signals
- Baseline → neutral

Should the ML prediction be corrected?

Return ONLY one label:
baseline, stress, amusement, meditation
"""
prompt = PromptTemplate(
    input_variables=[
        "eda_mean","eda_std","bvp_mean","bvp_std",
        "temp_mean","acc_mean","acc_std","ml_prediction"
    ],
    template=template
)

In [40]:
#This is the new method I use now. We may look into doing some more prompt engineering to see if that might elicit better results

def hybrid_predict(row, ml_pred):
    prompt_text = f"""
    You are an expert in emotion recognition using physiological signals.

    Features:
    EDA mean: {row['eda_mean']}
    EDA std: {row['eda_std']}
    BVP mean: {row['bvp_mean']}
    BVP std: {row['bvp_std']}
    Temp mean: {row['temp_mean']}
    ACC mean: {row['acc_mean']}
    ACC std: {row['acc_std']}

    ML model predicted: {ml_pred}

    Choose ONE:
    baseline, stress, amusement, meditation

    Only return the label.
    """
    
    response = llm.invoke(prompt_text)
    return response.strip().lower()

In [41]:
#this uses the above prompt to run through 50 samples and see how accurate it is. I limited it to 50 because it takes awhile
#for the LLM to respond but we will get a better metric of how correct it is with more samples

X_sample = X_test.iloc[:50]
y_sample = y_test.iloc[:50]
ml_sample = y_pred_labels[:50]

hybrid_preds = []

for i, (_, row) in enumerate(X_sample.iterrows()):
    pred = hybrid_predict(row, ml_sample[i])
    hybrid_preds.append(pred)

In [42]:
print("Hybrid Accuracy:", accuracy_score(y_sample, hybrid_preds))

print("\nConfusion Matrix:")
print(confusion_matrix(y_sample, hybrid_preds))

print("\nClassification Report:")
print(classification_report(y_sample, hybrid_preds))

Hybrid Accuracy: 0.78

Confusion Matrix:
[[ 8  0  0  0]
 [ 3  5  0  5]
 [ 1  0 17  2]
 [ 0  0  0  9]]

Classification Report:
              precision    recall  f1-score   support

   amusement       0.67      1.00      0.80         8
    baseline       1.00      0.38      0.56        13
  meditation       1.00      0.85      0.92        20
      stress       0.56      1.00      0.72         9

    accuracy                           0.78        50
   macro avg       0.81      0.81      0.75        50
weighted avg       0.87      0.78      0.77        50



In [43]:
#this gives an explanation of a sample from the set to see how the LLM is working through it, I may mess around with this more
#i'd like to give it a series of random samples that have previously printed their labels to see how accurate or correct the LLMs logic is
def explain_prediction(row, ml_pred):
    explanation_prompt = f"""
    Explain why this is {ml_pred} based on:
    EDA: {row['eda_mean']}
    BVP: {row['bvp_mean']}
    Temp: {row['temp_mean']}
    """
    return llm.invoke(explanation_prompt)

print(explain_prediction(X_sample.iloc[0], ml_sample[0]))

A fascinating topic!

The output you provided appears to be a summary of physiological signals collected during a meditation session, specifically:

1. **EDA (Electrodermal Activity)**: This measures the electrical activity of the skin, which is influenced by emotional arousal and relaxation responses. A low EDA value like 0.45283827500000007 indicates that the person was likely in a relaxed state, which is typical during meditation.
2. **BVP (Beat-to-Beat Variability)**: This measures the variations in heart rate from beat to beat, which can reflect physical and emotional states. A low BVP value like 0.4435468750000001 suggests that the person's heart rate was relatively stable and slow, consistent with a meditative state.
3. **Temp (Temperature)**: This is the body temperature, which can be influenced by various factors, including stress levels and relaxation responses. A normal body temperature of around 34.29599999999999°C (93.8°F) suggests that the person was in a relaxed state, w